In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, fisher_exact


1.Load data & Data Overview

In [45]:
df = pd.read_excel("../data/2022Cdata.xlsx")

In [46]:
print(df.shape)
df.info()
df.head(10)

(58, 5)
<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   文物编号    58 non-null     int64
 1   纹饰      58 non-null     str  
 2   类型      58 non-null     str  
 3   颜色      54 non-null     str  
 4   表面风化    58 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.4 KB


,文物编号,纹饰,类型,颜色,表面风化
0,1,C,高钾,蓝绿,无风化
1,2,A,铅钡,浅蓝,风化
2,3,A,高钾,蓝绿,无风化
3,4,A,高钾,蓝绿,无风化
4,5,A,高钾,蓝绿,无风化
5,6,A,高钾,蓝绿,无风化
6,7,B,高钾,蓝绿,风化
7,8,C,铅钡,紫,风化
8,9,B,高钾,蓝绿,风化
9,10,B,高钾,蓝绿,风化


2. Missing Value

In [47]:
df.isnull().sum()

文物编号    0
纹饰      0
类型      0
颜色      4
表面风化    0
dtype: int64

3. Value counts

In [48]:
for col in ["类型","纹饰","颜色","表面风化"]:

        print("="*20)
        print(col)
        print(df[col].value_counts(dropna=False))
        

类型
类型
铅钡    40
高钾    18
Name: count, dtype: int64
纹饰
纹饰
C    30
A    22
B     6
Name: count, dtype: int64
颜色
颜色
浅蓝     20
蓝绿     15
深绿      7
紫       4
NaN     4
浅绿      3
深蓝      2
黑       2
绿       1
Name: count, dtype: int64
表面风化
表面风化
风化     34
无风化    24
Name: count, dtype: int64


In [49]:
print(df.columns)

Index(['文物编号', '纹饰', '类型', '颜色', '表面风化'], dtype='str')


## 1. Cross-tabulation between Glass Type and Weathering

Purpose:
To summarize the frequency distribution of weathering status under different glass types.

In [50]:
table_type_display=pd.crosstab( 
    df["类型"], df["表面风化"] , margins=True)
table_type

表面风化,无风化,风化,All
类型,,,
铅钡,12,28,40
高钾,12,6,18
All,24,34,58


In [51]:
table_type=pd.crosstab( 
    df["类型"], df["表面风化"] )
table_type

表面风化,无风化,风化
类型,,
铅钡,12,28
高钾,12,6


In [52]:
chi2_contingency?

Signature: chi2_contingency(observed, correction=True, lambda_=None, *, method=None)
Docstring:
Chi-square test of independence of variables in a contingency table.

This function computes the chi-square statistic and p-value for the
hypothesis test of independence of the observed frequencies in the
contingency table [1]_ `observed`.  The expected frequencies are computed
based on the marginal sums under the assumption of independence; see
`scipy.stats.contingency.expected_freq`.  The number of degrees of
freedom is (expressed using numpy functions and attributes)::

    dof = observed.size - sum(observed.shape) + observed.ndim - 1

Parameters
----------
observed : array_like
    The contingency table. The table contains the observed frequencies
    (i.e. number of occurrences) in each category.  In the two-dimensional
    case, the table is often described as an "R x C table".
correction : bool, optional
    If True, *and* the degrees of freedom is 1, apply Yates' correction
    for c

In [53]:
result=chi2_contingency(
    table_type, 
    correction=False)
result


Chi2ContingencyResult(statistic=np.float64(6.8803921568627455), pvalue=np.float64(0.008714644061182652), dof=1, expected_freq=array([[16.55172414, 23.44827586],
       [ 7.44827586, 10.55172414]]))

In [54]:
print("Chi-square stastistic :", result.statistic )
print("p-value:",result.pvalue)
print( "degree of freedom:" , result.dof)
print("expected frequency:",result.expected_freq)

Chi-square stastistic : 6.8803921568627455
p-value: 0.008714644061182652
degree of freedom: 1
expected frequency: [[16.55172414 23.44827586]
 [ 7.44827586 10.55172414]]


In [55]:
expected = pd.DataFrame(
    result.expected_freq,
    index=table_type.index,
    columns=table_type.columns
)

expected

表面风化,无风化,风化
类型,,
铅钡,16.551724,23.448276
高钾,7.448276,10.551724


In [56]:
print("Minimum expected frequency:", expected.min().min())


Minimum expected frequency: 7.448275862068965


### Chi-square Assumption Check

All expected frequencies are greater than 5.

Therefore, the assumptions of the Pearson Chi-square test are satisfied.

Since p-value < 0.05,

the null hypothesis of independence is rejected.

Therefore,

glass type and weathering status are significantly associated.